# C12-classical-models — Session 3: Kernel SVM and Dual Intuition

*One 90-minute session. We reuse F7's $g\le0$, $\lambda\ge0$ convention, weak duality,
complementary slackness, and valid-kernel tests.*


In [ ]:
import numpy as np
from sklearn.datasets import make_circles
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

SEED = 20260804
ATOL = 1e-9
RTOL = 1e-7
rng = np.random.default_rng(SEED)


## 1. From the soft-margin primal to dual coefficients

Write constraints as $g_i=1-\xi_i-t_i(w^Tx_i+b)\le0$ with multiplier
$\alpha_i\ge0$, plus $-\xi_i\le0$. Stationarity in $w$ gives
$w=\sum_i\alpha_it_ix_i$, and stationarity in $b$ gives $\sum_i\alpha_it_i=0$.
The dual maximizes

$$\sum_i\alpha_i-\frac12\sum_{i,j}\alpha_i\alpha_jt_it_jx_i^Tx_j$$

subject to $0\le\alpha_i\le C$ and $\sum_i\alpha_it_i=0$ for the standard sum-loss primal.
Only dot products remain.

**Checkpoint 1A.** Which stationarity condition produces $\sum_i\alpha_it_i=0$?

**Checkpoint 1B.** What box constraint corresponds to soft-margin penalty `C`?


## 2. Complementary slackness and the support-vector decision function

The two inequality families are $1-\xi_i-t_if(x_i)\le0$ with multiplier
$\alpha_i\ge0$, and $-\xi_i\le0$ with multiplier $\beta_i\ge0$. Stationarity in slack gives
$C-\alpha_i-\beta_i=0$, so $0\le\alpha_i\le C$ and $\beta_i=C-\alpha_i$.
Complementary slackness gives

$$\alpha_i[1-\xi_i-t_if(x_i)]=0,\qquad \beta_i\xi_i=0.$$

These equations support three distinct, limited conclusions.

- If $\alpha_i=0$, then $\beta_i=C>0$ and hence $\xi_i=0$. The margin constraint may be
  strict or active, so this case permits $t_if(x_i)\ge1$; $\alpha_i=0$ alone does not prove a
  strict margin. Its coefficient vanishes, so the row is not a support vector in the decision sum.
- If $0<\alpha_i<C$, then $\beta_i>0$ forces $\xi_i=0$, while $\alpha_i>0$ forces the
  margin constraint active. Therefore $t_if(x_i)=1$: the row lies exactly on a margin plane.
- If $\alpha_i=C$, then $\beta_i=0$, and $\alpha_i>0$ still forces
  $t_if(x_i)=1-\xi_i\le1$. This includes an on-margin row when $\xi_i=0$, a correctly classified
  within-margin row when $0<\xi_i<1$, a boundary row when $\xi_i=1$, or a misclassified row
  when $\xi_i>1$. Thus $\alpha_i=C$ does not prove misclassification.

Substituting stationary $w$ yields
$f(x)=\sum_{i\in SV}\alpha_it_i x_i^Tx+b$. With a kernel, replace the dot product:
$f(x)=\sum_{i\in SV}\alpha_it_iK(x_i,x)+b$. In scikit-learn `SVC`, `support_` stores
training-row indices, `support_vectors_` the rows, `dual_coef_[0]` the signed coefficients
$\alpha_it_i$ in binary classification, and `intercept_[0]` is $b$.

**Checkpoint 2A.** Why do zero-$\alpha_i$ rows disappear at prediction time?

**Checkpoint 2B.** Which attribute preserves the original training indices?

**Checkpoint 2C.** State the strongest valid margin/slack conclusion in each of the three
coefficient cases, and name one invalid converse.


## 3. Linear, polynomial, and RBF kernels

The linear kernel is $K(x,z)=x^Tz$. A polynomial kernel is
$K(x,z)=(\gamma x^Tz+r)^d$ using `gamma`, `coef0=r`, and integer `degree=d`.
The radial-basis-function kernel is

$$K(x,z)=\exp(-\gamma\|x-z\|_2^2),\qquad\gamma>0.$$

F7's PSD closure rules certify these common choices for valid parameters. Large RBF `gamma`
makes narrow, local influence and can produce intricate boundaries; small `gamma` makes broad,
smooth influence. Kernel validity does not guarantee good generalization.

**Checkpoint 3A.** What is $K_{RBF}(x,x)$?

**Checkpoint 3B.** Qualitatively, what happens to locality as `gamma` increases?


In [ ]:
def rbf_kernel(X, Z, gamma):
    X = np.asarray(X, dtype=np.float64)
    Z = np.asarray(Z, dtype=np.float64)
    squared = ((X[:, None, :] - Z[None, :, :]) ** 2).sum(axis=2)
    return np.exp(-gamma * squared)

X_probe = np.array([[0., 0.], [1., 0.], [0., 2.]])
K_probe = rbf_kernel(X_probe, X_probe, gamma=0.5)
assert K_probe.shape == (3, 3)
assert np.allclose(np.diag(K_probe), 1.0, atol=ATOL, rtol=RTOL)
assert np.allclose(K_probe, K_probe.T, atol=ATOL, rtol=RTOL)


## 4. Scaling and the interaction of `C` and `gamma`

RBF distances combine feature coordinates. A feature measured in thousands can dominate one
measured in tenths, so fit `StandardScaler` inside the pipeline and cross-validation folds.
`C` controls the price of training violations; `gamma` controls the spatial reach of each support
vector. Large `C` plus large `gamma` is a high-flexibility corner and can overfit. Small `C` or
small `gamma` smooths by different mechanisms, so tune them jointly on a fixed grid.

Use `make_pipeline(StandardScaler(), SVC(...))`. For an auditable fit, pin `kernel`, `C`,
`gamma`, and `random_state` where probability calibration or randomized behavior uses it.

**Checkpoint 4A.** Why must scaling be fitted inside each training fold?

**Checkpoint 4B.** Which hyperparameter changes spatial reach rather than violation price?


## 5. Worked example: fit and reconstruct an RBF decision score

We fit a seeded circle dataset. The reconstruction uses the pipeline's scaled probe, fitted
support vectors (which live in scaled coordinates), signed dual coefficients, and intercept.
It must match `decision_function` independently of predicted labels. This audit catches a missing
label sign, wrong `gamma`, or forgotten intercept.

**Checkpoint 5A.** What shape does binary `dual_coef_` have with $S$ support vectors?

**Checkpoint 5B.** Why is matching `predict` weaker than matching the decision score?


In [ ]:
X_circle, y01 = make_circles(n_samples=80, factor=0.35, noise=0.04,
                                random_state=SEED)
t_circle = 2 * y01 - 1
model = make_pipeline(StandardScaler(), SVC(kernel="rbf", C=4.0, gamma=1.25))
model.fit(X_circle, t_circle)
probe = np.array([[0.0, 0.0], [0.8, 0.0], [-0.6, 0.3]], dtype=np.float64)
scaled_probe = model.named_steps["standardscaler"].transform(probe)
svc = model.named_steps["svc"]
manual_score = (svc.dual_coef_[0][:, None] *
                rbf_kernel(svc.support_vectors_, scaled_probe, svc._gamma)).sum(axis=0)
manual_score += svc.intercept_[0]
api_score = model.decision_function(probe)
assert np.allclose(manual_score, api_score, atol=1e-8, rtol=1e-7)
print("support vectors", len(svc.support_), "| scores", api_score)


## 6. Linear versus kernel SVM on the comparison axes

A linear SVM has global linear geometry and $D$ coefficients; prediction cost is essentially a
dot product. A kernel SVM has nonlinear capacity through similarities and prediction cost grows
with the number of support vectors. Both output decision scores rather than native probabilities,
both depend on scaling, and both tune `C` by validation. A kernel model adds kernel family and
parameters such as `gamma` or `degree`.

Probability estimates requested through `SVC(probability=True)` add a calibration procedure; do
not equate them with the raw margin. Fit calibration only on training/validation data.

**Checkpoint 6A.** Which model's prediction cost depends directly on support-vector count?

**Checkpoint 6B.** Is an RBF decision score a probability? Why not?


## 7. Pitfalls, exam connections, and forward comparison

**Pitfalls.** Forgetting the label sign in dual coefficients, computing kernels on unscaled probes,
confusing `gamma` with `C`, treating every training row as a support vector, comparing scores
after omitting $b$, and tuning on test data all violate the fitted contract.

**Exam connection.** Expect a small support-vector ledger: compute a kernel row, multiply by signed
dual coefficients, add the intercept, and apply the stated zero-score tie. Alternatively audit a
pipeline and explain `C`/`gamma` effects without claiming one always increases accuracy.

**Going deeper.** Session 4 adds axis-aligned, scale-insensitive tree geometry and direct rule
interpretability to the same comparison ledger.

**Checkpoint 7A.** Name the four quantities needed to reconstruct a kernel-SVM score.

**Checkpoint 7B.** Give one reason a valid kernel can still generalize poorly.


## Checkpoint answers

**1A.** Stationarity with respect to $b$. **1B.** $0\le\alpha_i\le C$.

**2A.** Their coefficient multiplies the kernel term by zero. **2B.** `support_`.
**2C.** $\alpha=0$ gives $\xi=0$ and margin at least 1 but does not prove a strict margin;
$0<\alpha<C$ gives $\xi=0$ and margin exactly 1; $\alpha=C$ gives margin $1-\xi\le1$
but does not prove misclassification, because $\xi$ may range from zero upward.

**3A.** 1. **3B.** Influence becomes more local/narrow.

**4A.** Otherwise validation rows influence means/scales and leak information. **4B.** `gamma`.

**5A.** `(1, S)`. **5B.** Many wrong scores can share the same sign and therefore the same label.

**6A.** Kernel SVM. **6B.** No; it is an unrestricted signed margin score, not normalized to
$[0,1]$.

**7A.** Support vectors, signed dual coefficients, kernel parameters/function, and intercept.
**7B.** Hyperparameters can make the boundary too flexible, or the kernel geometry can mismatch
the task.
